# 🧪 7회차 실습: RAG & 멀티 에이전트

AI에게 **나만의 지식**을 가르치고, 여러 에이전트가 **협업**하는 시스템을 만들어봅니다!

In [ ]:
import os, sys
sys.path.append("..")
from dotenv import find_dotenv, load_dotenv
from openai import OpenAI
import chromadb
from collections import deque

load_dotenv(find_dotenv(usecwd=True), override=True)

APIM_BASE_URL = os.environ["APIM_BASE_URL"].rstrip("/")
APIM_KEY      = os.environ["APIM_KEY"]
MODEL         = os.getenv("CHAT_MODEL", "gpt-5.4")
EMBED         = os.getenv("EMBEDDING_MODEL", "text-embedding-3-small")

# Chat 클라이언트
client = OpenAI(
    api_key="placeholder",
    base_url=f"{APIM_BASE_URL}/{MODEL}/",
    default_headers={"api-key": APIM_KEY},
)

# Embedding 클라이언트 (모델 경로가 다름)
embed_client = OpenAI(
    api_key="placeholder",
    base_url=f"{APIM_BASE_URL}/{EMBED}/",
    default_headers={"api-key": APIM_KEY},
)

print("✅ 준비 완료!")
print(f"  CHAT      = {MODEL}")
print(f"  EMBEDDING = {EMBED}")


## 실습 1: 메모리가 있는 챗봇 🧠

In [2]:
class MemoryChatbot:
    def __init__(self, persona, max_history=10):
        self.persona = persona
        self.history = deque(maxlen=max_history)
        self.summary = ""

    def chat(self, user_input):
        messages = [{"role": "system", "content": self.persona + (f"\n[이전 대화 요약] {self.summary}" if self.summary else "")}]
        messages.extend(list(self.history))
        messages.append({"role": "user", "content": user_input})
        r = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.7, max_completion_tokens=500)
        reply = r.choices[0].message.content
        self.history.append({"role": "user", "content": user_input})
        self.history.append({"role": "assistant", "content": reply})
        return reply

bot = MemoryChatbot("너는 친절한 AI 비서야. 사용자의 정보를 기억하고 대화해.", max_history=6)
print("🤖", bot.chat("안녕! 나는 민수야. 고1이고 축구를 좋아해."))
print()
print("🤖", bot.chat("내 이름이 뭐라고 했지?"))

🤖 안녕 민수야! 만나서 반가워. 고1이고 축구를 좋아한다고 했지? 어떤 포지션에서 주로 뛰는 거야?

🤖 너의 이름은 민수야! 축구 좋아한다고도 기억하고 있어. 혹시 요즘 축구 연습 어떻게 하고 있어?


## 실습 2: RAG 시스템 구축 📚

### Step 1: 문서 준비

In [3]:
documents = [
    {"id": "s1", "content": "한빛고등학교는 2010년에 개교, 서울시 강남구 위치. 전교생 약 900명, 교사 60명.", "source": "학교소개"},
    {"id": "s2", "content": "급식은 매일 점심 제공. 월=한식, 화=양식, 수=중식, 목=일식, 금=특별메뉴.", "source": "급식"},
    {"id": "s3", "content": "동아리 활동은 매주 수요일 7교시. 인기 동아리: 코딩반, 밴드부, 축구부, 독서토론반.", "source": "동아리"},
    {"id": "s4", "content": "중간고사 4월 셋째주, 기말고사 7월 첫째주. 수행평가는 수시 진행.", "source": "시험일정"},
    {"id": "s5", "content": "교복: 하복(5~9월), 동복(10~4월). 매월 마지막 금요일은 사복의 날.", "source": "교복"},
    {"id": "s6", "content": "도서관 평일 8시~21시 운영. 대출 1인 최대 5권, 기간 2주.", "source": "도서관"},
    {"id": "s7", "content": "수업 중 휴대폰 금지. 점심시간과 방과 후에만 허용.", "source": "생활규정"},
    {"id": "s8", "content": "코딩반은 수요일 7교시 컴퓨터실(본관 3층). 지도교사 김철수 선생님, Python과 AI 다룸. 부원 25명.", "source": "코딩반"},
]
print(f"📄 문서 {len(documents)}개 준비 완료")

📄 문서 8개 준비 완료


### Step 2: 벡터 DB 저장

In [4]:
chroma = chromadb.Client()
collection = chroma.create_collection(name="school_info")

for doc in documents:
    emb = embed_client.embeddings.create(model=EMBED, input=[doc["content"]]).data[0].embedding
    collection.add(ids=[doc["id"]], embeddings=[emb], documents=[doc["content"]], metadatas=[{"source": doc["source"]}])

print(f"✅ {collection.count()}개 문서 저장 완료!")

✅ 8개 문서 저장 완료!


### Step 3: 검색 + 답변

In [5]:
def rag_answer(question):
    q_emb = embed_client.embeddings.create(model=EMBED, input=[question]).data[0].embedding
    results = collection.query(query_embeddings=[q_emb], n_results=3)
    context = "\n".join([f"[{results['metadatas'][0][i]['source']}] {results['documents'][0][i]}" for i in range(len(results['documents'][0]))])
    r = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "너는 한빛고 안내 AI야. 제공된 문서 기반으로만 답변해. 없는 정보는 '찾을 수 없습니다'라고 답해."},
            {"role": "user", "content": f"참고 문서:\n{context}\n\n질문: {question}"}
        ],
        temperature=0.3, max_completion_tokens=500
    )
    return r.choices[0].message.content

print("🤖", rag_answer("코딩반은 언제 활동해?"))
print()
print("🤖", rag_answer("도서관에서 책 몇 권 빌릴 수 있어?"))
print()
print("🤖", rag_answer("교장선생님 이름이 뭐야?"))

🤖 코딩반은 매주 수요일 7교시에 활동합니다.

🤖 도서관에서 1인 최대 5권까지 책을 빌릴 수 있습니다.

🤖 찾을 수 없습니다.


## 실습 3: 멀티 에이전트 토론 🎭

In [6]:
class DebateAgent:
    def __init__(self, name, position, style):
        self.name = name; self.position = position; self.style = style
    def speak(self, topic, prev):
        context = "\n".join(prev[-4:]) if prev else "(첫 발언)"
        r = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": f"너는 '{self.name}' 토론자. 입장: {self.position}. 말투: {self.style}. 3문장 이내로 발언해."},
                {"role": "user", "content": f"주제: {topic}\n이전 발언:\n{context}\n\n발언해줘."}
            ],
            temperature=0.8, max_completion_tokens=200
        )
        return r.choices[0].message.content

def run_debate(topic, rounds=2):
    pro = DebateAgent("찬성이", "찬성", "논리적, 데이터 인용")
    con = DebateAgent("반대이", "반대", "감성적, 실제 사례")
    args = []
    print(f"🎙️ 토론 주제: {topic}\n{'='*50}")
    for r in range(rounds):
        print(f"\n--- 라운드 {r+1} ---")
        ps = pro.speak(topic, args); args.append(f"[찬성] {ps}"); print(f"\n👍 찬성이: {ps}")
        cs = con.speak(topic, args); args.append(f"[반대] {cs}"); print(f"\n👎 반대이: {cs}")
    summary = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": f"토론 정리해줘.\n주제: {topic}\n{'\n'.join(args)}"}],
        max_completion_tokens=300
    )
    print(f"\n{'='*50}\n👨‍⚖️ 사회자:\n{summary.choices[0].message.content}")

run_debate("고등학교에서 AI 사용을 허용해야 하는가?")

🎙️ 토론 주제: 고등학교에서 AI 사용을 허용해야 하는가?

--- 라운드 1 ---

👍 찬성이: 고등학교에서 AI 사용을 허용하는 것은 학생들의 학습 효율성을 극대화하는 데 필수적입니다. 예를 들어, AI 도구는 맞춤형 학습을 지원하여 학생들의 이해도를 30% 이상 향상시킨다는 연구 결과가 있습니다. 따라서 AI 활용은 미래 교육 혁신의 핵심 요소입니다.

👎 반대이: AI가 학생들의 창의력과 사고력을 빼앗는 현실을 외면할 수 없습니다. 제 친구 중 한 명은 AI에 의존하다가 스스로 문제를 해결하는 능력을 잃어버려 대학 입시에서 큰 어려움을 겪었어요. 우리는 기술보다 인간의 성장을 더 소중히 여겨야 합니다.

--- 라운드 2 ---

👍 찬성이: AI는 단순한 도구일 뿐이며, 학생 스스로 활용하는 능력을 기르는 것이 중요합니다. 실제로 AI 활용 교육을 받은 학생들의 문제 해결 능력이 25% 이상 향상되었다는 연구가 있습니다. 따라서 AI 허용은 창의력과 사고력 증진에 오히려 긍정적 영향을 미칩니다.

👎 반대이: 제 아들이 AI에 너무 의존하다가 스스로 생각할 기회를 잃고, 결국 중요한 시험에서 실패했던 경험이 있습니다. AI가 모든 답을 주면 학생들은 고민하고 성장할 기회를 빼앗깁니다. 우리는 기술보다 학생들의 내면적 성장을 더 지켜줘야 하지 않을까요?

👨‍⚖️ 사회자:
토론 정리: 고등학교에서 AI 사용 허용에 대한 찬반 의견

1. 찬성 입장
- AI 사용은 학생들의 학습 효율성을 크게 향상시킴.
- 맞춤형 학습 지원으로 학생들의 이해도가 30% 이상 증가했다는 연구 결과가 있음.
- AI 활용 교육은 문제 해결 능력을 25% 이상 향상시키며, 창의력과 사고력 증진에 긍정적 영향.
- AI는 단순한 도구이며, 학생들이 이를 능동적으로 활용하는 능력을 기르는 것이 중요.
- AI는 미래 교육 혁신의 핵심 요소로 자리매김할 것으로 기대됨.

2. 반대 입장
- AI 의존은 학생들의 창의력과 사고력을 저하시킬 위험이 있음.
- AI에 지나치게 의존한